In [1]:
import os
import random
import numpy as np
from nltk.corpus import stopwords
from nltk import word_tokenize
from string import punctuation
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Embedding, Conv1D, MaxPooling1D, Dropout, Flatten, Dense, concatenate)

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)
2026-07-31 19:32:47.323718: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
os.listdir('./data/neg/')

['cv000_29416.txt',
 'cv001_19502.txt',
 'cv002_17424.txt',
 'cv003_12683.txt',
 'cv004_12641.txt',
 'cv005_29357.txt',
 'cv006_17022.txt',
 'cv007_4992.txt',
 'cv008_29326.txt',
 'cv009_29417.txt',
 'cv010_29063.txt',
 'cv011_13044.txt',
 'cv012_29411.txt',
 'cv013_10494.txt',
 'cv014_15600.txt',
 'cv015_29356.txt',
 'cv016_4348.txt',
 'cv017_23487.txt',
 'cv018_21672.txt',
 'cv019_16117.txt',
 'cv020_9234.txt',
 'cv021_17313.txt',
 'cv022_14227.txt',
 'cv023_13847.txt',
 'cv024_7033.txt',
 'cv025_29825.txt',
 'cv026_29229.txt',
 'cv027_26270.txt',
 'cv028_26964.txt',
 'cv029_19943.txt',
 'cv030_22893.txt',
 'cv031_19540.txt',
 'cv032_23718.txt',
 'cv033_25680.txt',
 'cv034_29446.txt',
 'cv035_3343.txt',
 'cv036_18385.txt',
 'cv037_19798.txt',
 'cv038_9781.txt',
 'cv039_5963.txt',
 'cv040_8829.txt',
 'cv041_22364.txt',
 'cv042_11927.txt',
 'cv043_16808.txt',
 'cv044_18429.txt',
 'cv045_25077.txt',
 'cv046_10613.txt',
 'cv047_18725.txt',
 'cv048_18380.txt',
 'cv049_21917.txt',
 'cv050_

In [3]:
s = str.maketrans('', '', punctuation)
a = 'Drood!@'
a.translate(s)

'Drood'

In [4]:
stop_words = stopwords.words('english')
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [5]:
negative_docs = []
max_len_negative = 0
for file in os.listdir('./data/neg'):
    with open('./data/neg/' + file) as f:
        text = f.read()
        tokens = word_tokenize(text)
        translator = str.maketrans('', '', punctuation)
        tokens = [w.translate(translator) for w in tokens]
        tokens = [w for w in tokens if not w in stop_words]
        if len(tokens) > max_len_negative:
            max_len_negative = len(tokens)

        negative_docs.append(' '.join(tokens))

len(negative_docs)

1000

In [6]:
positive_docs = []
max_len_positive = 0
for file in os.listdir('./data/pos'):
    with open('./data/pos/' + file) as f:
        text = f.read()
        tokens = word_tokenize(text)
        translator = str.maketrans('', '', punctuation)
        tokens = [w.translate(translator) for w in tokens]
        tokens = [w for w in tokens if not w in stop_words]
        if len(tokens) > max_len_positive:
            max_len_positive = len(tokens)

        positive_docs.append(' '.join(tokens))

len(positive_docs)

1000

In [7]:
max_len = max(max_len_negative, max_len_positive)

In [8]:
random.shuffle(negative_docs)
random.shuffle(positive_docs)

In [9]:
X_train = negative_docs[:800] + positive_docs[:800]
y_train = [0 for _ in range(800)] + [1 for _ in range(800)]

In [10]:
X_test = negative_docs[800:] + positive_docs[800:]
y_test = [0 for _ in range(len(negative_docs) - 800)] + [1 for _ in range(len(positive_docs) - 800)]

In [11]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

In [12]:
vocab_len = len(tokenizer.word_index) + 1
vocab_len

40657

In [13]:
encoded = tokenizer.texts_to_sequences(X_train)
encoded_test = tokenizer.texts_to_sequences(X_test)

In [14]:
padded = pad_sequences(encoded, maxlen=max_len, padding='post')
padded_test = pad_sequences(encoded_test, maxlen=max_len, padding='post')

In [15]:
padded.shape

(1600, 1693)

In [16]:
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(vocab_len, 200),
    Conv1D(filters=64, kernel_size=4, activation='relu'),
    MaxPooling1D(2),
    Dropout(0.5),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

2026-07-31 19:33:07.176886: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
model.compile(
    optimizer='Adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 1693, 200)      │     8,131,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1690, 64)       │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 845, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 845, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 54080)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,461,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,645,961 (44.43 MB)

 Trainable params: 11,645,961 (44.43 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.fit(
    x=padded,
    y=np.array(y_train),
    validation_data=(padded_test, np.array(y_test)),
    batch_size=20,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=15),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.2, min_lr=0.001),
        tf.keras.callbacks.ModelCheckpoint('./model/chestxray.h5', monitor='val_loss', mode='min', save_best_only=True)
    ]
)

Epoch 1/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.5060 - loss: 0.6967

80/80 ━━━━━━━━━━━━━━━━━━━━ 35s 399ms/step - accuracy: 0.5131 - loss: 0.6935 - val_accuracy: 0.5625 - val_loss: 0.6835 - learning_rate: 0.0010
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step - accuracy: 0.6998 - loss: 0.5807

80/80 ━━━━━━━━━━━━━━━━━━━━ 30s 380ms/step - accuracy: 0.7644 - loss: 0.4973 - val_accuracy: 0.7125 - val_loss: 0.5429 - learning_rate: 0.0010
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 30s 371ms/step - accuracy: 0.9806 - loss: 0.0679 - val_accuracy: 0.6975 - val_loss: 0.6741 - learning_rate: 0.0010
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step - accuracy: 1.0000 - loss: 0.0049

80/80 ━━━━━━━━━━━━━━━━━━━━ 30s 370ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 0.8175 - val_loss: 0.4161 - learning_rate: 0.0010
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 28s 346ms/step - accuracy: 1.0000 - loss: 0.0015 - val_accuracy: 0.8050 - val_loss: 0.4350 - learning_rate: 0.0010
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 28s 353ms/step - accuracy: 1.0000 - loss: 8.8080e-04 - val_accuracy: 0.8150 - val_loss: 0.4222 - learning_rate: 0.0010
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 29s 365ms/step - accuracy: 1.0000 - loss: 5.8818e-04 - val_accuracy: 0.8225 - val_loss: 0.4282 - learning_rate: 0.0010
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 29s 357ms/step - accuracy: 1.0000 - loss: 3.4157e-04 - val_accuracy: 0.8200 - val_loss: 0.4194 - learning_rate: 0.0010
Epoch 9/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 29s 359ms/step - accuracy: 1.0000 - loss: 2.5128e-04 - val_accuracy: 0.8075 - val_loss: 0.4427 - learning_rate: 0.0010
Epoch 10/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 30s 374ms/step - accuracy: 1.0000 - lo

In [20]:
model.save('model/textcnn.h5')

## Now we'll try another text and give it to our model and see how much the model can predict the polarity of that text

In [21]:
from tensorflow.keras.models import load_model

In [22]:
model = load_model('model/textcnn.h5')

In [23]:
# Positive text
text1 = '''
I rarely write reviews, but this experience truly deserved one. From the very first moment I arrived, I noticed how organized, welcoming, and professional everything was. It immediately gave me confidence that I had made the right choice.
The staff members were exceptionally friendly and polite. Every question I asked was answered with patience, and they genuinely seemed interested in making sure I had a great experience. Their positive attitude made a significant difference.
The environment itself was clean, modern, and very comfortable. Everything looked well maintained, and it was obvious that attention had been paid to even the smallest details. It created a relaxing atmosphere that made me feel at home.
One of the highlights was the outstanding quality of the service. Nothing felt rushed or careless. Every step was handled efficiently, and I never had to wait longer than expected. Everything worked exactly as promised.
I was also impressed by the consistency. Sometimes businesses start strong but fail to maintain their standards. In this case, every interaction was just as pleasant as the previous one, which made the entire experience even more enjoyable.
The quality exceeded my expectations in every possible way. The products were excellent, the presentation was attractive, and everything felt carefully prepared. It was clear that quality and customer satisfaction were top priorities.
Another thing I appreciated was the honesty and transparency. There were no hidden costs, confusing explanations, or misleading information. Everything was communicated clearly, which made me trust the company even more.
I genuinely felt valued as a customer. Small gestures, friendly conversations, and professional behavior showed that they cared about providing an exceptional experience instead of simply completing a transaction.
After leaving, I couldn't stop thinking about how enjoyable everything had been. It is becoming increasingly rare to find businesses that combine professionalism, kindness, efficiency, and high quality so successfully.
I have already recommended this place to several friends and family members because I believe they deserve the same wonderful experience. I am confident they will be just as satisfied as I was.
Without any hesitation, I would gladly return again in the future. This experience exceeded all my expectations and reminded me how enjoyable outstanding customer service can be.
Overall, I would describe this as an exceptional, memorable, reliable, enjoyable, and highly satisfying experience. It is one of the best experiences I have had in a long time, and I would rate it five out of five without any doubt.
'''

In [24]:
# Negative text
text2 = '''
I honestly wish I had never chosen this place because the entire experience was disappointing from beginning to end. Very little went as expected, and every stage introduced a new problem that made the situation even more frustrating.
The first thing I noticed was the poor attitude of the staff. They seemed uninterested, impatient, and unwilling to answer even the simplest questions. Instead of making customers feel welcome, they made me feel like an inconvenience.
The environment was far below acceptable standards. It was noisy, poorly organized, and not particularly clean. Several areas looked neglected, which immediately reduced my confidence in the quality of the service.
Unfortunately, the problems continued throughout my visit. Everything took much longer than expected, and there was almost no communication about the delays. Waiting without any explanation became increasingly frustrating.
The overall quality was extremely poor. Nothing met the expectations that had been created beforehand. It felt as though very little effort had been invested in delivering a professional experience.
To make matters worse, several mistakes occurred during the process. Important details were overlooked, parts of my request were ignored, and correcting those mistakes required additional time and effort on my part.
What disappointed me most was the complete lack of accountability. Instead of apologizing or attempting to solve the problems, the employees acted as though everything was normal and that customer satisfaction was unimportant.
The value for the money was terrible. Considering the amount I paid, I expected a much higher standard of quality and professionalism. Instead, I left feeling that I had completely wasted both my time and my money.
Even after leaving, I continued thinking about how poorly everything had been handled. The experience left a lasting negative impression that could easily have been avoided with better management and more attentive customer service.
I cannot recommend this place to anyone. There are many better alternatives that offer higher quality, friendlier service, and much greater attention to customer satisfaction.
I certainly have no intention of returning in the future. One disappointing experience might be understandable, but this contained too many problems to overlook or excuse.
Overall, this was an unpleasant, stressful, frustrating, unprofessional, and completely unsatisfactory experience. It failed to meet even my most basic expectations, and I would strongly advise others to avoid it.
'''

In [25]:
tokens = word_tokenize(text=text1)
translator = str.maketrans('', '', punctuation)
tokens = [w.translate(translator) for w in tokens]
tokens = [w for w in tokens if not w in stop_words]
pos_text = ' '.join(tokens)

In [26]:
tokens = word_tokenize(text=text2)
translator = str.maketrans('', '', punctuation)
tokens = [w.translate(translator) for w in tokens]
tokens = [w for w in tokens if not w in stop_words]
neg_text = ' '.join(tokens)

In [27]:
encoder1 = tokenizer.texts_to_sequences([pos_text])
pos_text = pad_sequences(encoder1, maxlen=max_len, padding='post')

encoder2 = tokenizer.texts_to_sequences([neg_text])
neg_text = pad_sequences(encoder2, maxlen=max_len, padding='post')

In [28]:
neg_text.shape

(1, 1693)

In [29]:
pred1 = model.predict(pos_text)
if float(pred1) > 0.5:
    print('The text is positive')
else:
    print('The text is negative')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
The text is positive


/tmp/ipykernel_9861/3510046741.py:2: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if float(pred1) > 0.5:


In [30]:
pred2 = model.predict(neg_text)
if float(pred2) > 0.5:
    print('The text is positive')
else:
    print('The text is negative')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
The text is negative


/tmp/ipykernel_9861/2130013025.py:2: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if float(pred2) > 0.5:
